# 17 — Physiological-pH Protonation / Net Formal Charge Diagnostic

**This is a cheap diagnostic only.** No model training, no submission, no architecture work. Its
sole purpose is to gate whether a later, expensive descriptor-augmented Chemprop run (charge +
logP proxy concatenated as `X_d` to `chemprop_chemeleoninit`) is worth building at all.

## Why charge, specifically

A D-MPNN encodes the molecular graph. Molecular weight, HBA, HBD, ring count and stereocentre
count are all direct functions of that graph, so handing them to the model as extra descriptors
is largely redundant. Protonation state at pH 7.4 is **not** recoverable from a neutral input
graph — it is the one candidate descriptor that is qualitatively unlike anything the encoder
already sees.

Mechanistically this matters most for **CYP2D6**, which binds via an ionic interaction between a
protonated basic nitrogen and Asp301, and secondarily for **CYP2C9**, whose pocket has a cationic
Arg108 and whose classic ligands (warfarin, diclofenac, sulfaphenazole, tolbutamide) are acids.
The a-priori expectation is CYP2D6 positive and CYP2C9 negative on a signed net-charge column.
CYP1A2 (narrow planar pocket, mostly neutral aromatics) and CYP3A4 (large promiscuous pocket, no
strong electrostatic preference) are expected to show little.

Kiani & Jabeen (2019), already in this project's reference library, selected "total charge" as
the **branching node** of their CYP2D6 decision tree. Their Methods state physicochemical
properties including formal charges came from FAF-Drugs4, and their Table 3 reports a mean total
charge of 0.67 across all data (0.69 actives, 0.29 inactives). Neutral-SMILES formal charge would
sit at approximately zero for the great majority of drug-like compounds, so their feature is
evidently a pH-aware protonated-state charge — that 0.67 is the numerical anchor Part 2 checks
against. Their descriptor **selection** is treated as weak evidence, not established fact: their
own Table 4 reports 10-fold CV AUCs of 0.64 / 0.502 / 0.589 / 0.543 for CYP1A2 / CYP2C9 / CYP2D6 /
CYP3A4, with the high accuracy figures driven by severe class imbalance. This notebook tests the
mechanism against **this project's own data**, not their conclusion.

## Decisions already made (not re-litigated here)

- Protonation tool: **Dimorphite-DL**, pH 7.4, **dominant microstate only**. If it cannot be
  installed or fails to run, the task instructions say stop and ask rather than silently falling
  back to RDKit's neutral-SMILES formal charge — that substitution would produce a
  near-meaningless column for exactly the isoform this whole check is about.
- Net charge = sum of formal charges over the dominant protonated microstate, as a signed integer.
- logP: RDKit Crippen `MolLogP`, computed on the **neutral** canonical SMILES (matching
  `src/features.py`'s own `isoform_structural_descriptors` precedent). Labelled explicitly
  throughout as a **proxy** for FAF-Drugs4's logD(pH7), not an equivalent — never presented as
  logD.

## A real environment wrinkle, resolved and documented before this notebook was built

`pip install dimorphite-dl` (as instructed) pulls in `rdkit<2026` as a hard dependency, which
directly conflicts with this project's `rdkit==2026.3.3` pin (required by chemprop's
`cuik-molmaker-pin` dependency chain) — installing it **silently downgraded** the live
`cyp-admet-v2` environment's RDKit to 2025.9.6. This was caught and fixed immediately (before any
of the work below ran): `pip install --no-deps --force-reinstall rdkit==2026.3.3` restored the
pinned version, verified afterward by confirming `import dimorphite_dl` still works against it and
that chemprop can still featurize a molecule end-to-end via `cuik_molmaker`. Full detail, including
why this can't be expressed as a normal one-line pip pin in `environment.yml`, is recorded there
(new comment block, dated 2026-09-14) and the pre-change spec is archived at
`environment/archive/environment_cyp-admet-v2_2026-09-14_pre-dimorphite-dl.yml`.

## Scope boundaries

Does not touch `data/folds/cv_folds.csv`, `outputs/05_cv_comparison/`,
`outputs/board_solved_calibration/`, `src/calibration.py`, or any existing submission artifact.
Does not train, retrain, or fine-tune anything. Does not touch or make any claim about blind-set
labels (there are none). Does not modify `src/features.py` — no genuinely new shared
canonicalisation/InChIKey logic was needed here, only trivial one-line RDKit/Dimorphite-DL calls,
which is not what that module exists to centralise.

## 0. Setup

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))  # so `from src... import ...` works regardless of cwd

import importlib.metadata
import json

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Crippen
from scipy.stats import spearmanr

from dimorphite_dl import protonate_smiles
from src.vendor.openadmet_eval.config import REGRESSION_ENDPOINTS

print(f"python: {sys.version.split()[0]}")
for pkg in ["rdkit", "numpy", "pandas", "scipy", "dimorphite-dl"]:
    print(f"{pkg}: {importlib.metadata.version(pkg)}")
print(f"REGRESSION_ENDPOINTS: {REGRESSION_ENDPOINTS}")

PROCESSED = REPO_ROOT / "data" / "processed"
OUT = REPO_ROOT / "outputs" / "17_protonation_charge_check"
OUT.mkdir(parents=True, exist_ok=True)

PH = 7.4

python: 3.11.13
rdkit: 2026.3.3
numpy: 1.26.4
pandas: 2.3.3
scipy: 1.17.1
dimorphite-dl: 2.0.2
REGRESSION_ENDPOINTS: ['CYP1A2_pIC50_direct_inhibition', 'CYP2C9_pIC50_direct_inhibition', 'CYP2D6_pIC50_direct_inhibition', 'CYP3A4_pIC50_direct_inhibition']


## Part 1 — Verify the protonation is real before computing anything else

Dimorphite-DL's `protonate_smiles` enumerates every microstate whose population is significant
within a `[ph_min, ph_max]` window (controlled by `precision`). Setting `ph_min == ph_max == 7.4`
with `precision=0.0` collapses this to exactly one state per ionisable site — the side of that
site's own pKa that pH 7.4 falls on — which is what "dominant microstate only" means operationally
here. Confirmed empirically below and, later, across the full curated dataset: this setting always
returns exactly one SMILES per input compound, never zero, never more than one.

Six literature compounds are spot-checked: four well-known CYP2D6-relevant basic amines
(propranolol, quinidine, desipramine, metoprolol), each with a nitrogen expected to be protonated
(net charge **+1**) at pH 7.4, and two well-known acids (diclofenac, warfarin), each expected to be
deprotonated (net charge **-1**). Their reference SMILES are hardcoded below (standard structures
for these compounds) — they are not drawn from this project's curated dataset, since the point is
an independent literature-based sanity check, not a self-consistency check against data this
notebook will go on to use.

In [2]:
def dominant_microstate(smiles, ph=PH):
    # Returns (dominant_microstate_smiles, net_formal_charge) at `ph`, or (None, None) if
    # protonation fails outright (unparseable input) or -- defensively, never actually observed
    # below or across the full dataset in Part 2 -- returns something other than exactly one
    # microstate.
    try:
        variants = protonate_smiles(smiles, ph_min=ph, ph_max=ph, precision=0.0)
    except Exception:
        return None, None
    if len(variants) != 1:
        return None, None
    micro_smiles = variants[0]
    mol = Chem.MolFromSmiles(micro_smiles)
    if mol is None:
        return micro_smiles, None
    return micro_smiles, Chem.GetFormalCharge(mol)


SPOT_CHECK = [
    # name, input SMILES, role, expected net charge at pH 7.4
    ("propranolol", "CC(C)NCC(O)COc1cccc2ccccc12", "basic amine", 1),
    ("quinidine", "CC[C@H]1CN2CC[C@H]1C[C@@H]2[C@H](O)c1ccnc2ccc(OC)cc12", "basic amine", 1),
    ("desipramine", "CNCCCN1c2ccccc2CCc2ccccc21", "basic amine", 1),
    ("metoprolol", "COCCc1ccc(OCC(O)CNC(C)C)cc1", "basic amine", 1),
    ("diclofenac", "OC(=O)Cc1ccccc1Nc1c(Cl)cccc1Cl", "carboxylic acid", -1),
    ("warfarin", "CC(=O)CC(c1ccccc1)c1c(O)c2ccccc2oc1=O", "carboxylic acid (enol)", -1),
]

rows = []
for name, smi, role, expected in SPOT_CHECK:
    micro_smiles, charge = dominant_microstate(smi)
    rows.append({
        "compound": name, "role": role, "input_smiles": smi,
        "dominant_microstate_smiles": micro_smiles, "net_charge_ph74": charge,
        "expected": expected, "pass": charge == expected,
    })
spot_check_df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 60)
print(spot_check_df.to_string(index=False))

all_pass = bool(spot_check_df["pass"].all())
print()
if all_pass:
    print("PART 1 VERIFICATION: PASS -- all 6/6 compounds protonate to their expected sign. "
          "Proceeding to Part 2.")
else:
    failed = spot_check_df.loc[~spot_check_df["pass"], "compound"].tolist()
    raise RuntimeError(
        f"PART 1 VERIFICATION: FAIL -- protonation did not match expectation for {failed}. "
        "Stopping per task instructions rather than proceeding on a meaningless charge column."
    )

   compound                   role                                          input_smiles                                dominant_microstate_smiles  net_charge_ph74  expected  pass
propranolol            basic amine                           CC(C)NCC(O)COc1cccc2ccccc12                          CC(C)[NH2+]CC(O)COc1cccc2ccccc12                1         1  True
  quinidine            basic amine CC[C@H]1CN2CC[C@H]1C[C@@H]2[C@H](O)c1ccnc2ccc(OC)cc12 CC[C@H]1C[NH+]2CC[C@H]1C[C@@H]2[C@H](O)c1ccnc2ccc(OC)cc12                1         1  True
desipramine            basic amine                            CNCCCN1c2ccccc2CCc2ccccc21                           C[NH2+]CCCN1c2ccccc2CCc2ccccc21                1         1  True
 metoprolol            basic amine                           COCCc1ccc(OCC(O)CNC(C)C)cc1                          COCCc1ccc(OCC(O)C[NH2+]C(C)C)cc1                1         1  True
 diclofenac        carboxylic acid                        OC(=O)Cc1ccccc1Nc1c(Cl)cccc1Cl            

**Result: PASS, 6/6.** All four basic amines (propranolol, quinidine, desipramine, metoprolol)
protonate to net charge **+1** at their expected basic nitrogen (for quinidine, correctly the
aliphatic quinuclidine nitrogen, not the weakly-basic aromatic quinoline nitrogen — chemically the
right site). Both acids (diclofenac, warfarin) deprotonate to net charge **-1**. Protonation is
genuinely being applied, not a no-op returning neutral structures — safe to proceed.

## Part 2 — Compute for the full curated dataset

`canonical_smiles` and `inchikey` are read directly from the already-curated CSVs rather than
recomputed — those columns were themselves produced by `src/features.py`'s
`add_canonical_smiles_and_inchikey` during curation (notebook 01), so reusing them here **is** the
"go through the shared module, don't recompute" rule in practice, not a deviation from it.

In [3]:
train_df = pd.read_csv(PROCESSED / "train_inhibition_curated.csv")
test_df = pd.read_csv(PROCESSED / "test_blinded_curated.csv")
print(f"loaded train_inhibition_curated.csv: {train_df.shape}")
print(f"loaded test_blinded_curated.csv: {test_df.shape}")

loaded train_inhibition_curated.csv: (4905, 20)
loaded test_blinded_curated.csv: (750, 4)


### 2.1 Compute net charge (pH 7.4) and Crippen logP for every compound in both splits

logP is computed on the **neutral** canonical SMILES (via `Crippen.MolLogP`), matching
`src/features.py`'s `isoform_structural_descriptors` convention for its own `logd_proxy` field —
LogP by definition refers to the neutral species; this is deliberately labelled a proxy for
logD(pH7), never presented as logD itself. Net charge is computed on the dominant protonated
microstate from Part 1's `dominant_microstate` function (reused as-is, not reimplemented) applied
per-row. Any compound where protonation or the neutral-SMILES parse fails is flagged, not dropped.

In [4]:
def compute_split(df, label):
    n = len(df)
    print(f"{label}: {n} compounds before computation")
    micro_list, charge_list, logp_list = [], [], []
    failures = []
    for _, row in df.iterrows():
        smi = row["canonical_smiles"]
        micro_smiles, charge = dominant_microstate(smi)
        mol_neutral = Chem.MolFromSmiles(smi)
        logp = Crippen.MolLogP(mol_neutral) if mol_neutral is not None else np.nan
        if charge is None or mol_neutral is None:
            failures.append((row["Molecule_Name"], smi))
        micro_list.append(micro_smiles)
        charge_list.append(charge)
        logp_list.append(logp)
    out = df[["inchikey", "Molecule_Name", "canonical_smiles"]].copy()
    out["dominant_microstate_smiles"] = micro_list
    out["net_charge_ph74"] = charge_list
    out["crippen_logp"] = logp_list
    out["split"] = label
    print(f"{label}: {n} compounds after computation, {len(failures)} failed")
    return out, failures


train_feat, train_failures = compute_split(train_df, "train")
test_feat, test_failures = compute_split(test_df, "test")

train: 4905 compounds before computation


train: 4905 compounds after computation, 0 failed
test: 750 compounds before computation


test: 750 compounds after computation, 0 failed


### 2.2 Sanity statistics and failure report

In [5]:
print("=== Failures ===")
print(f"train: {len(train_failures)} failed")
for name, smi in train_failures[:5]:
    print(f"  {name}: {smi}")
print(f"test: {len(test_failures)} failed")
for name, smi in test_failures[:5]:
    print(f"  {name}: {smi}")

print()
print("=== net_charge_ph74 / crippen_logp sanity stats ===")
combined_feat = pd.concat([train_feat, test_feat], ignore_index=True)
for label, sub in [("train", train_feat), ("test", test_feat), ("combined", combined_feat)]:
    charges = sub["net_charge_ph74"].dropna()
    logps = sub["crippen_logp"].dropna()
    total = len(charges)
    n_pos = int((charges == 1).sum())
    n_zero = int((charges == 0).sum())
    n_neg = int((charges == -1).sum())
    n_other = total - n_pos - n_zero - n_neg
    print(f"\n-- {label} (n={len(sub)}) --")
    print(f"net_charge_ph74: min={charges.min()}, max={charges.max()}, mean={charges.mean():.4f}")
    print("value_counts:")
    print(charges.value_counts().sort_index(ascending=False).to_string())
    print(f"fractions: -1={n_neg/total:.4f}  0={n_zero/total:.4f}  "
          f"+1={n_pos/total:.4f}  other={n_other/total:.4f}")
    print(f"crippen_logp: min={logps.min():.3f}, max={logps.max():.3f}, mean={logps.mean():.4f}")

=== Failures ===
train: 0 failed
test: 0 failed

=== net_charge_ph74 / crippen_logp sanity stats ===

-- train (n=4905) --
net_charge_ph74: min=-5, max=3, mean=0.3741
value_counts:
net_charge_ph74
 3      19
 2     358
 1    1743
 0    2173
-1     548
-2      61
-3       2
-5       1
fractions: -1=0.1117  0=0.4430  +1=0.3554  other=0.0899
crippen_logp: min=-3.358, max=13.586, mean=2.7925

-- test (n=750) --
net_charge_ph74: min=-3, max=3, mean=0.1333
value_counts:
net_charge_ph74
 3      4
 2     28
 1    169
 0    430
-1    102
-2     16
-3      1
fractions: -1=0.1360  0=0.5733  +1=0.2253  other=0.0653
crippen_logp: min=-0.195, max=6.211, mean=2.8345

-- combined (n=5655) --
net_charge_ph74: min=-5, max=3, mean=0.3422
value_counts:
net_charge_ph74
 3      23
 2     386
 1    1912
 0    2603
-1     650
-2      77
-3       3
-5       1
fractions: -1=0.1149  0=0.4603  +1=0.3381  other=0.0866
crippen_logp: min=-3.358, max=13.586, mean=2.7981


**Result: zero failures** in both splits (0/4905 train, 0/750 test) — every curated SMILES, being
already RDKit-canonical from curation, parses and protonates cleanly; no RDKit warnings were
emitted during either split's run. Training-set net charge spans -5 to +3 (mean **0.374**), with a
substantial fraction carrying nonzero charge: 11.2% at -1, 35.5% at +1, 9.0% at some other nonzero
value — **55.7% of training compounds are charged** at pH 7.4 under this protonation. The blind
test set shows a similar but somewhat less-charged shape (mean 0.133; 57.3% neutral vs. train's
44.3%) — expected population variation between the two sets, not something this notebook
investigates further (no blind labels exist to relate it to). Crippen logP ranges are unremarkable
and typical of drug-like compounds (train mean 2.79, test mean 2.83).

### 2.3 Comparison to Kiani & Jabeen (2019)'s reported mean charge (0.67)

In [6]:
train_mean_charge = train_feat["net_charge_ph74"].mean()
diff = train_mean_charge - 0.67
print(f"This project's training-set mean net_charge_ph74: {train_mean_charge:.4f}")
print(f"Kiani & Jabeen (2019) Table 3 reported mean total charge: 0.67")
print(f"difference: {diff:+.4f}")

This project's training-set mean net_charge_ph74: 0.3741
Kiani & Jabeen (2019) Table 3 reported mean total charge: 0.67
difference: -0.2959


**Result: 0.374, versus Kiani's 0.67 (difference -0.296).** This sits just below the 0.4-0.9 range
this task pre-stated as "consistent with their population," but it is **not** the near-zero value
that would indicate protonation silently failed to apply — Part 1's spot checks already confirmed
the mechanism directly, and 55.7% of training compounds carry nonzero charge (2.2 above), which a
broken/no-op protonation could not produce. The gap is more plausibly explained by this project's
compound population differing from Kiani's own dataset, and/or Dimorphite-DL's pKa predictions
differing from FAF-Drugs4's, than by a computation error. Per the task instructions, nothing is
tuned to close this gap — it is reported and left as-is.

### 2.4 Neutral-SMILES formal charge vs. protonated net charge (training set only)

Quantifies the size of the bug this whole check exists to guard against: what would this feature
have looked like if Dimorphite-DL had silently been skipped in favour of plain RDKit
`GetFormalCharge` on the as-given (neutral) SMILES?

In [7]:
train_feat["net_charge_neutral"] = train_df["canonical_smiles"].apply(
    lambda smi: Chem.GetFormalCharge(Chem.MolFromSmiles(smi))
)

n_compared = len(train_feat)
n_diff = int((train_feat["net_charge_ph74"] != train_feat["net_charge_neutral"]).sum())
print(f"training compounds compared: {n_compared}")
print(f"compounds where net_charge_ph74 != net_charge_neutral: "
      f"{n_diff} ({n_diff / n_compared * 100:.2f}%)")
print()
print("net_charge_neutral value_counts (training set):")
print(train_feat["net_charge_neutral"].value_counts().sort_index(ascending=False).to_string())

training compounds compared: 4905
compounds where net_charge_ph74 != net_charge_neutral: 2716 (55.37%)

net_charge_neutral value_counts (training set):
net_charge_neutral
 1      11
 0    4884
-1      10


**Result: 2,716 / 4,905 training compounds (55.4%) differ** between the protonated and neutral
charge columns. The neutral-SMILES column is almost entirely degenerate — 4,884/4,905 (99.6%) sit
at exactly 0, with only 21 compounds carrying any nonzero formal charge as drawn. This is precisely
the "near-meaningless column for exactly the isoform this whole check is about" the task
instructions warned a silent RDKit-only fallback would produce: had that substitution happened,
Part 3's per-isoform correlations below would have been computed against a column that is constant
at zero for 99.6% of the data.

### 2.5 Save reusable feature file

In [8]:
combined_feat = pd.concat([
    train_feat[["inchikey", "Molecule_Name", "canonical_smiles", "net_charge_ph74",
                "net_charge_neutral", "crippen_logp", "split"]],
    test_feat.assign(net_charge_neutral=np.nan)[
        ["inchikey", "Molecule_Name", "canonical_smiles", "net_charge_ph74",
         "net_charge_neutral", "crippen_logp", "split"]
    ],
], ignore_index=True)

csv_path = OUT / "charge_logp_features.csv"
combined_feat.to_csv(csv_path, index=False)
print(f"saved {csv_path} ({combined_feat.shape[0]} rows, {combined_feat.shape[1]} columns)")
print(combined_feat.head(3).to_string(index=False))

saved /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/17_protonation_charge_check/charge_logp_features.csv (5655 rows, 7 columns)
                   inchikey Molecule_Name           canonical_smiles  net_charge_ph74  net_charge_neutral  crippen_logp split
RADKZDMFGJYCBB-UHFFFAOYSA-N  OCNT-0000422        Cc1ncc(CO)c(C=O)c1O               -1                 0.0       0.40042 train
TXCGAZHTZHNUAI-UHFFFAOYSA-N  OCNT-0001882 CC(C)(Oc1ccc(Cl)cc1)C(=O)O               -1                 0.0       2.58200 train
XBJWOGLKABXFJE-UHFFFAOYSA-N  OCNT-0007477      CCOC(=O)N1CSCC1C(=O)O                0                 0.0       0.60240 train


`net_charge_neutral` is populated for training rows only, `NaN` for test rows — matching Part 2.4's
explicit training-set-only scope (the blind set's neutral-charge column was never computed; this is
deliberate, not a missing-data bug, and is spelled out here so it isn't mistaken for one). This
file is reusable infrastructure, kept separate from any model-specific output, per task scope.

## Pre-registered decision criteria (frozen before Part 3's results are computed)

Per this project's threshold-discipline convention (CLAUDE.md: CV-measured/descriptor-signal
comparisons need an explicit, pre-stated resolution threshold frozen before looking at results —
the same discipline as notebook 09/11b's placement-correction pursuit threshold). These criteria
are written and saved to disk in this cell, **before** any Spearman correlation or group-mean table
below is computed.

- **PASS** (worth carrying into a later Chemprop run): `|Spearman rho| >= 0.10` on at least one
  isoform, **OR** a difference in median pIC50 between the +1 and 0 charge groups of `>= 0.3` log
  units on at least one isoform, with both groups having `n >= 100`.
- **MECHANISM CONFIRMED** (a stronger result): CYP2D6's correlation is positive **and** CYP2C9's is
  negative, **both** meeting the PASS bar above.
- **FAIL**: neither of the above.

Verdicts are reported per isoform without averaging across isoforms to force one overall verdict —
the whole premise of this check is that isoforms are expected to differ.

In [9]:
DECISION_CRITERIA = {
    "notebook": "17_protonation_charge_check",
    "frozen_before": "Part 3 per-isoform Spearman / group-mean results",
    "pass_rule": {
        "spearman_abs_threshold": 0.10,
        "median_diff_charge1_vs_0_threshold": 0.3,
        "median_diff_min_group_n": 100,
        "logic": "PASS if EITHER condition holds on at least one isoform",
    },
    "mechanism_confirmed_rule": (
        "CYP2D6 Spearman rho positive AND CYP2C9 Spearman rho negative, "
        "BOTH independently meeting the PASS spearman_abs_threshold"
    ),
    "fail_rule": "neither pass_rule condition holds on any isoform",
    "isoforms_evaluated_independently": True,
    "no_cross_isoform_averaging": True,
}

criteria_path = OUT / "decision_criteria.json"
with open(criteria_path, "w") as f:
    json.dump(DECISION_CRITERIA, f, indent=2)
print(f"saved {criteria_path}")
print(json.dumps(DECISION_CRITERIA, indent=2))

saved /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/17_protonation_charge_check/decision_criteria.json
{
  "notebook": "17_protonation_charge_check",
  "frozen_before": "Part 3 per-isoform Spearman / group-mean results",
  "pass_rule": {
    "spearman_abs_threshold": 0.1,
    "median_diff_charge1_vs_0_threshold": 0.3,
    "median_diff_min_group_n": 100,
    "logic": "PASS if EITHER condition holds on at least one isoform"
  },
  "mechanism_confirmed_rule": "CYP2D6 Spearman rho positive AND CYP2C9 Spearman rho negative, BOTH independently meeting the PASS spearman_abs_threshold",
  "fail_rule": "neither pass_rule condition holds on any isoform",
  "isoforms_evaluated_independently": true,
  "no_cross_isoform_averaging": true
}


## Part 3 — Does charge carry signal, per isoform?

Training set only (the blind test set has no labels). Spearman is used here strictly as a
**diagnostic** of whether a raw descriptor carries any signal against the label — it is **not** a
model-selection metric. This project's governing metric is MA-ST-RAE, which does not apply to a
single descriptor column. Any downstream model comparison built on this feature must be scored on
OOF ST-RAE with the vendored evaluator, not on Spearman.

For each isoform, rows are filtered to that isoform's own non-null pIC50 subset (the same
per-isoform-subset convention used throughout this project, e.g. notebooks 05b/05c) — row counts
before/after are logged explicitly.

In [10]:
train_merged = train_feat.merge(
    train_df[["inchikey"] + REGRESSION_ENDPOINTS], on="inchikey", how="left"
)

def charge_bucket(c):
    return str(int(c)) if c in (-1, 0, 1) else "other"

train_merged["charge_bucket"] = train_merged["net_charge_ph74"].apply(charge_bucket)

results = {}
for endpoint in REGRESSION_ENDPOINTS:
    isoform = endpoint.split("_")[0]
    n_before = int(train_merged[endpoint].notna().sum())
    sub = train_merged.dropna(subset=[endpoint, "net_charge_ph74"])
    n_after = len(sub)
    print(f"\n{'=' * 20} {isoform} {'=' * 20}")
    print(f"row count: {n_before} labelled before charge join, {n_after} after "
          f"(charge/logp never failed, so these should match)")

    rho_charge, p_charge = spearmanr(sub["net_charge_ph74"], sub[endpoint])
    print(f"\nnet_charge_ph74 vs {endpoint}: Spearman rho={rho_charge:.4f}, p={p_charge:.4g}, "
          f"n={n_after}")

    print("\nprimary 4-bucket table (-1 / 0 / +1 / other):")
    bucket_grp = (
        sub.groupby("charge_bucket")[endpoint]
        .agg(["mean", "median", "count"])
        .reindex(["-1", "0", "1", "other"])
    )
    print(bucket_grp.to_string())

    print("\nfull per-integer-charge breakdown (supplementary detail):")
    full_grp = sub.groupby("net_charge_ph74")[endpoint].agg(["mean", "median", "count"])
    print(full_grp.to_string())

    rho_logp, p_logp = spearmanr(sub["crippen_logp"], sub[endpoint])
    print(f"\n[secondary reference] crippen_logp vs {endpoint}: Spearman rho={rho_logp:.4f}, "
          f"p={p_logp:.4g}, n={n_after}")

    median_1 = bucket_grp.loc["1", "median"] if "1" in bucket_grp.index else np.nan
    median_0 = bucket_grp.loc["0", "median"] if "0" in bucket_grp.index else np.nan
    n_1 = bucket_grp.loc["1", "count"] if "1" in bucket_grp.index else 0
    n_0 = bucket_grp.loc["0", "count"] if "0" in bucket_grp.index else 0
    median_diff = median_1 - median_0 if pd.notna(median_1) and pd.notna(median_0) else np.nan

    results[isoform] = {
        "n": n_after, "rho_charge": rho_charge, "p_charge": p_charge,
        "rho_logp": rho_logp, "p_logp": p_logp,
        "median_diff_1_vs_0": median_diff, "n_group1": int(n_1), "n_group0": int(n_0),
    }

print("\n\n=== summary ===")
summary_df = pd.DataFrame(results).T
print(summary_df.to_string())


==================== CYP1A2 ====================
row count: 1412 labelled before charge join, 1412 after (charge/logp never failed, so these should match)

net_charge_ph74 vs CYP1A2_pIC50_direct_inhibition: Spearman rho=-0.2047, p=7.921e-15, n=1412

primary 4-bucket table (-1 / 0 / +1 / other):
                   mean    median  count
charge_bucket                           
-1             5.092089  5.225218    199
0              5.103683  5.216009    756
1              4.691801  4.921999    392
other          4.402204  4.552174     65

full per-integer-charge breakdown (supplementary detail):
                     mean    median  count
net_charge_ph74                           
-5               5.249564  5.249564      1
-3               5.396836  5.396836      2
-2               4.920508  5.158043     14
-1               5.092089  5.225218    199
 0               5.103683  5.216009    756
 1               4.691801  4.921999    392
 2               4.145676  3.944095     45
 3         

### Per-isoform read (with real computed numbers)

| Isoform | Spearman rho (charge) | n | Spearman rho (logP) |
|---|---|---|---|
| CYP1A2 | **-0.2047** (p=8e-15) | 1,412 | +0.2329 |
| CYP2C9 | **-0.1535** (p=3e-8) | 1,285 | +0.4853 |
| CYP2D6 | **+0.1691** (p=5e-11) | 1,493 | +0.0342 (n.s., p=0.19) |
| CYP3A4 | -0.0712 (p=6e-4) | 2,335 | +0.6020 |

None of the four isoforms clears the +1-vs-0 median-pIC50-diff arm of the PASS rule on its own
(closest: CYP1A2 at 0.294 log units, just under the 0.3 bar); every PASS verdict below is carried
by the Spearman arm alone.

**A confound worth checking before over-reading CYP1A2's result**: net charge and Crippen logP are
themselves anti-correlated in this population (Spearman rho = -0.1024, p=7e-13, n=4,905) —
chemically unsurprising (charged species are less lipophilic). Where an isoform's logP correlation
with pIC50 is itself strongly positive, this confound alone predicts a *negative* charge-pIC50
correlation with no electrostatic mechanism required. CYP1A2 (logp rho +0.23) and CYP2C9 (logp rho
+0.49) both fit this pattern — their negative charge correlations are **consistent with, and not
distinguishable from, being driven by the charge/logP confound** rather than a direct electrostatic
effect. **CYP2D6 is the exception**: its logP correlation with pIC50 is essentially zero and
non-significant (rho +0.034, p=0.19) — the confound predicts *no* charge signal here, yet a real
positive one (+0.169) is observed anyway. CYP2D6's signal is the one that survives this check;
CYP1A2's and CYP2C9's do not cleanly rule out the simpler lipophilicity explanation.

## Part 4 (continued) — Applying the frozen criteria to Part 3's actual results

In [11]:
with open(criteria_path) as f:
    criteria = json.load(f)

spearman_thresh = criteria["pass_rule"]["spearman_abs_threshold"]
median_thresh = criteria["pass_rule"]["median_diff_charge1_vs_0_threshold"]
min_n = criteria["pass_rule"]["median_diff_min_group_n"]

print(f"Frozen thresholds: |Spearman| >= {spearman_thresh}, OR median diff >= {median_thresh} "
      f"with both groups n >= {min_n}\n")

verdicts = {}
for isoform, r in results.items():
    spearman_pass = abs(r["rho_charge"]) >= spearman_thresh
    median_pass = (
        pd.notna(r["median_diff_1_vs_0"])
        and abs(r["median_diff_1_vs_0"]) >= median_thresh
        and r["n_group1"] >= min_n
        and r["n_group0"] >= min_n
    )
    isoform_pass = bool(spearman_pass or median_pass)
    verdicts[isoform] = isoform_pass
    print(f"{isoform}: rho={r['rho_charge']:+.4f} (spearman_pass={spearman_pass}), "
          f"median_diff={r['median_diff_1_vs_0']:+.4f} "
          f"[n1={r['n_group1']}, n0={r['n_group0']}] (median_pass={median_pass}) "
          f"-> {'PASS' if isoform_pass else 'FAIL'}")

overall_pass = bool(any(verdicts.values()))
mechanism_confirmed = bool(
    results["CYP2D6"]["rho_charge"] > 0 and abs(results["CYP2D6"]["rho_charge"]) >= spearman_thresh
    and results["CYP2C9"]["rho_charge"] < 0 and abs(results["CYP2C9"]["rho_charge"]) >= spearman_thresh
)

print(f"\nOverall (any isoform passes): {'PASS' if overall_pass else 'FAIL'}")
print(f"MECHANISM CONFIRMED (CYP2D6 positive AND CYP2C9 negative, both >= threshold): "
      f"{mechanism_confirmed}")

verdict_summary = {
    "per_isoform_pass": verdicts,
    "overall_pass": overall_pass,
    "mechanism_confirmed": mechanism_confirmed,
}
verdict_path = OUT / "verdict_summary.json"
with open(verdict_path, "w") as f:
    json.dump(verdict_summary, f, indent=2)
print(f"\nsaved {verdict_path}")

Frozen thresholds: |Spearman| >= 0.1, OR median diff >= 0.3 with both groups n >= 100

CYP1A2: rho=-0.2047 (spearman_pass=True), median_diff=-0.2940 [n1=392, n0=756] (median_pass=False) -> PASS
CYP2C9: rho=-0.1535 (spearman_pass=True), median_diff=-0.2190 [n1=456, n0=558] (median_pass=False) -> PASS
CYP2D6: rho=+0.1691 (spearman_pass=True), median_diff=+0.1665 [n1=627, n0=583] (median_pass=False) -> PASS
CYP3A4: rho=-0.0712 (spearman_pass=False), median_diff=-0.2004 [n1=811, n0=1033] (median_pass=False) -> FAIL

Overall (any isoform passes): PASS
MECHANISM CONFIRMED (CYP2D6 positive AND CYP2C9 negative, both >= threshold): True

saved /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/17_protonation_charge_check/verdict_summary.json


### Stated verdict, per isoform, per the criteria frozen above Part 3 — not softened either way

- **CYP1A2: PASS** (rho = -0.2047, clears the 0.10 bar comfortably — in fact the largest-magnitude
  charge correlation of any isoform). This was **not** the a-priori expectation (CYP1A2 was
  expected to show little signal) and, per the confound check above, is not distinguishable from a
  lipophilicity effect riding on the charge/logP anti-correlation rather than a direct electrostatic
  one.
- **CYP2C9: PASS** (rho = -0.1535, negative as mechanistically predicted). Also not distinguishable
  from the lipophilicity confound (logP rho +0.49 for this isoform).
- **CYP2D6: PASS** (rho = +0.1691, positive as mechanistically predicted) — **and the one isoform
  whose signal is not explained away by the charge/logP confound**, since logP itself carries
  essentially no relationship to CYP2D6 pIC50 here.
- **CYP3A4: FAIL** (rho = -0.0712, below the 0.10 bar; median diff 0.20, below the 0.3 bar) —
  matches the a-priori expectation of little signal for this isoform.

**MECHANISM CONFIRMED: TRUE** per the letter of the pre-registered rule (CYP2D6 positive and
CYP2C9 negative, both independently clearing the Spearman bar). Reported as frozen-criteria-met,
together with the caveat immediately above: CYP2D6's half of this result is the strand best
supported as a genuine electrostatic effect rather than a lipophilicity proxy; CYP2C9's half of it
is real (statistically) but not cleanly separable from that same confound. Isoforms are not
averaged together to force a single verdict — CYP3A4 fails outright, and that is reported as
plainly as the other three isoforms' passes.

## Part 5 — Report, and stop

No model was built, no architecture was modified, no retraining occurred, and nothing was
submitted. This section states the verdict and its implication for the gated decision — it does
not act on it.

**Overall verdict: PASS.** Three of four isoforms (CYP1A2, CYP2C9, CYP2D6) clear the pre-registered
`|Spearman| >= 0.10` bar, and the stronger MECHANISM CONFIRMED criterion is also met. Net formal
charge at physiological pH carries real, statistically significant signal against pIC50 in this
project's own curated data — this is not a null result.

**Does this support proceeding to a shared-descriptor Chemprop run** (charge + logP proxy
concatenated as `X_d` to `chemprop_chemeleoninit`)? On the aggregate PASS/FAIL bar alone: yes, the
diagnostic clears the bar this task set out to gate. But the confound check above means that bar
is not equally well-earned across isoforms:

- **CYP2D6** is the cleanest case — its positive charge signal survives the lipophilicity-confound
  check and matches the mechanistic prediction (protonated basic N / Asp301) this task set out to
  test. This is the isoform where a charge descriptor is most likely to be adding genuinely new
  information the graph encoder cannot already infer.
- **CYP1A2 and CYP2C9** pass the frozen numerical bar but their charge signal is not distinguishable
  from a byproduct of charge correlating with logP, a descriptor the encoder can already largely
  infer from the graph. Their inclusion in a future run would need to be judged on OOF ST-RAE, not
  assumed to help just because Spearman cleared the bar here.
- **CYP3A4** shows no signal by either registered criterion and matches its a-priori "little
  expected" prediction — no case for including charge here on this evidence.

**A downstream model comparison, if built, must be scored on OOF ST-RAE with the vendored
evaluator** (per Part 3's own framing) — Spearman here is a screening diagnostic, not a
model-selection metric, and nothing about today's PASS verdict predicts how a 4-isoform multitask
Chemprop model with an added `X_d` branch would actually score. That remains untested and is
explicitly out of scope for this notebook.

## Addendum — Quantifying the Charge/logP Confound via Partial Correlation

Part 3's own prose identified a real confound *qualitatively*: net charge and Crippen logP are
anti-correlated in this population (whole-training-set Spearman rho = -0.1024, p=7e-13, n=4,905),
so on any isoform where logP correlates strongly positively with pIC50, that anti-correlation
alone predicts a negative charge-pIC50 correlation with no electrostatic mechanism required. That
argument was convincing as far as it went, but it was never quantified — three isoforms currently
hold a Part 3 PASS verdict whose interpretation depends on it. This section settles it numerically
via Spearman **partial** correlation (charge vs. pIC50, controlling for logP), computed directly
from the standard first-order partial-correlation formula on rank-transformed (i.e. Spearman)
pairwise correlations:

$$r_{xy.z} = \frac{r_{xy} - r_{xz} \, r_{yz}}{\sqrt{(1 - r_{xz}^2)(1 - r_{yz}^2)}}$$

where $x$=`net_charge_ph74`, $y$=pIC50, $z$=`crippen_logp`, and all three pairwise $r$ are Spearman
on the same isoform subset. This is read-only analysis reusing already-computed columns — no
protonation, charge, or logP value is recomputed, no model is trained, and nothing is submitted.
`train_merged` (built in Part 3, still in scope in this same kernel session) is reused directly
rather than reloading `charge_logp_features.csv` or the curated labels from disk.

### Pre-registered interpretation (frozen before any partial correlation is computed)

Appended as a new, clearly-labelled second block in the existing
`outputs/17_protonation_charge_check/decision_criteria.json` — Part 3's original frozen block is
read back unmodified and left exactly as it was; nothing in it is overwritten.

- **CONFOUND-DRIVEN**: an isoform's partial `|rho|` drops below the same 0.10 bar already frozen
  for Part 3, having cleared it unconditionally. Its charge signal is then attributable to
  lipophilicity, and there is no case for a charge descriptor on that isoform.
- **SURVIVES**: partial `|rho|` stays at or above 0.10 with the same sign as the unconditional
  value. The charge signal is then independent of lipophilicity on that isoform.
- **A-priori expectation, stated now so it cannot be fitted afterwards**: CYP2D6 is expected to
  **SURVIVE** (its logP correlation is null — Part 3: rho=+0.0342, p=0.19, n.s. — so there is
  nothing for the confound to act through). CYP1A2 and CYP2C9 are genuinely uncertain and could go
  either way. CYP3A4 already FAILED Part 3 (unconditional `|rho|` below the 0.10 bar to begin
  with) and is reported below for completeness only — it never cleared the bar this label pair is
  about clearing-then-dropping-below, so neither label applies to it.

In [12]:
PARTIAL_CORRELATION_CRITERIA = {
    "notebook": "17_protonation_charge_check (addendum, appended same day)",
    "frozen_before": "any partial-correlation computation in this section",
    "purpose": (
        "quantify, via Spearman partial correlation controlling for crippen_logp, whether each "
        "Part-3-PASS isoform's charge-pIC50 correlation is attributable to the charge/logP "
        "anti-correlation Part 3 identified qualitatively, rather than an independent effect"
    ),
    "threshold_reused_from_part3": spearman_thresh,
    "labels": {
        "CONFOUND_DRIVEN": (
            "partial |rho| drops below the Part-3 threshold, having cleared it unconditionally "
            "-- charge signal attributable to lipophilicity, no case for a charge descriptor on "
            "that isoform"
        ),
        "SURVIVES": (
            "partial |rho| stays >= the Part-3 threshold, same sign as the unconditional value "
            "-- charge signal independent of lipophilicity on that isoform"
        ),
        "not_applicable": (
            "isoform already FAILED Part 3's unconditional bar -- reported for completeness "
            "only, not re-adjudicated with either label"
        ),
    },
    "a_priori_expectation_stated_before_computing": {
        "CYP2D6": "SURVIVES -- logP correlation is null, nothing for the confound to act through",
        "CYP1A2": "genuinely uncertain, could go either way",
        "CYP2C9": "genuinely uncertain, could go either way",
        "CYP3A4": "not_applicable -- already FAILED Part 3, reported for completeness only",
    },
}

with open(criteria_path) as f:
    existing_criteria = json.load(f)

if "partial_correlation_confound_check" in existing_criteria:
    raise RuntimeError(
        "decision_criteria.json already has a 'partial_correlation_confound_check' block -- "
        "stopping rather than silently overwriting a prior run of this section."
    )

existing_criteria["partial_correlation_confound_check"] = PARTIAL_CORRELATION_CRITERIA
with open(criteria_path, "w") as f:
    json.dump(existing_criteria, f, indent=2)

print(f"appended 'partial_correlation_confound_check' block to {criteria_path}")
print("Part 3's original keys, confirmed still present and unmodified:",
      [k for k in existing_criteria if k != "partial_correlation_confound_check"])
print()
print(json.dumps(PARTIAL_CORRELATION_CRITERIA, indent=2))

appended 'partial_correlation_confound_check' block to /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/17_protonation_charge_check/decision_criteria.json
Part 3's original keys, confirmed still present and unmodified: ['notebook', 'frozen_before', 'pass_rule', 'mechanism_confirmed_rule', 'fail_rule', 'isoforms_evaluated_independently', 'no_cross_isoform_averaging']

{
  "notebook": "17_protonation_charge_check (addendum, appended same day)",
  "frozen_before": "any partial-correlation computation in this section",
  "purpose": "quantify, via Spearman partial correlation controlling for crippen_logp, whether each Part-3-PASS isoform's charge-pIC50 correlation is attributable to the charge/logP anti-correlation Part 3 identified qualitatively, rather than an independent effect",
  "threshold_reused_from_part3": 0.1,
  "labels": {
    "CONFOUND_DRIVEN": "partial |rho| drops below the Part-3 threshold, having cleared it unconditionally -- charge signal attributable to lipophilici

### Computing the partial correlation, per isoform

`rho_charge` (unconditional charge-vs-pIC50) and `rho_logp` (logP-vs-pIC50) are **reused directly
from Part 3's `results` dict, not recomputed** — only the charge-vs-logP pairwise correlation is
newly computed here, on each isoform's own non-null-pIC50 subset (same per-isoform-subset
convention Part 3 used: `train_merged.dropna(subset=[endpoint, "net_charge_ph74"])`). A t-based
significance test for the partial correlation (df = n-3, the standard approach for a first-order
partial correlation) is reported alongside for context, using `scipy.stats.t` — already imported
via `scipy.stats`, no new dependency.

In [13]:
from scipy.stats import t as t_dist

partial_results = {}
for endpoint in REGRESSION_ENDPOINTS:
    isoform = endpoint.split("_")[0]
    sub = train_merged.dropna(subset=[endpoint, "net_charge_ph74", "crippen_logp"])
    n = len(sub)
    assert n == results[isoform]["n"], (
        f"{isoform}: partial-correlation subset n={n} does not match Part 3's stored n="
        f"{results[isoform]['n']} -- crippen_logp was assumed never missing where charge/pIC50 "
        f"are present; stopping rather than silently computing on a different population."
    )

    r_xy = results[isoform]["rho_charge"]  # charge vs pIC50, reused from Part 3
    r_yz = results[isoform]["rho_logp"]    # logp vs pIC50, reused from Part 3
    r_xz, p_xz = spearmanr(sub["net_charge_ph74"], sub["crippen_logp"])  # charge vs logp, new

    denom = np.sqrt((1 - r_xz**2) * (1 - r_yz**2))
    r_partial = (r_xy - r_xz * r_yz) / denom

    df = n - 3
    t_stat = r_partial * np.sqrt(df / (1 - r_partial**2))
    p_partial = 2 * (1 - t_dist.cdf(abs(t_stat), df))

    shift = r_partial - r_xy

    partial_results[isoform] = {
        "n": n,
        "rho_charge_unconditional": r_xy,
        "rho_logp": r_yz,
        "rho_charge_logp_this_subset": r_xz,
        "p_charge_logp_this_subset": p_xz,
        "rho_charge_partial": r_partial,
        "p_charge_partial": p_partial,
        "shift_partial_minus_unconditional": shift,
    }

partial_df = pd.DataFrame(partial_results).T
print(partial_df.to_string())

             n  rho_charge_unconditional  rho_logp  rho_charge_logp_this_subset  p_charge_logp_this_subset  rho_charge_partial  p_charge_partial  shift_partial_minus_unconditional
CYP1A2  1412.0                 -0.204734  0.232884                    -0.075012               4.799698e-03           -0.193104      2.551293e-13                           0.011631
CYP2C9  1285.0                 -0.153531  0.485297                    -0.100525               3.072217e-04           -0.120410      1.517182e-05                           0.033121
CYP2D6  1493.0                  0.169111  0.034173                    -0.164093               1.789032e-10            0.177223      5.409229e-12                           0.008112
CYP3A4  2335.0                 -0.071178  0.602046                    -0.114398               2.970684e-08           -0.002906      8.884122e-01                           0.068272


### Why charge-vs-logP differs from Part 3's whole-training-set figure (-0.1024)

The `rho_charge_logp_this_subset` column above is computed on each isoform's own labelled
subset (n=1,285-2,335), not the full 4,905-compound training set Part 3's headline -0.1024 figure
used — different, smaller, non-random (isoform-labelled) populations naturally shift a
correlation somewhat. This is expected population variation, not an inconsistency between this
section and Part 3, and is stated here explicitly so it isn't mistaken for one.

### Applying the pre-registered labels

In [14]:
partial_verdicts = {}
for isoform, r in partial_results.items():
    # `verdicts` is Part 4's own already-computed per-isoform PASS/FAIL dict (cell above,
    # spearman_pass OR median_pass) -- reused directly rather than re-derived from the spearman
    # threshold alone, so this section can never silently diverge from Part 3/4's actual verdict.
    part3_pass = bool(verdicts[isoform])
    same_sign = np.sign(r["rho_charge_partial"]) == np.sign(r["rho_charge_unconditional"])
    survives = bool(abs(r["rho_charge_partial"]) >= spearman_thresh and same_sign)

    if not part3_pass:
        label = "not_applicable"
        note = "already FAILED Part 3's unconditional bar -- reported for completeness only"
    elif survives:
        label = "SURVIVES"
        note = "partial |rho| stays >= threshold, same sign -- independent of logP"
    else:
        label = "CONFOUND_DRIVEN"
        note = "partial |rho| drops below threshold -- attributable to logP"

    partial_verdicts[isoform] = {
        "part3_pass": part3_pass,
        "label": label,
        "note": note,
        "rho_charge_unconditional": r["rho_charge_unconditional"],
        "rho_charge_partial": r["rho_charge_partial"],
    }
    print(f"{isoform}: unconditional={r['rho_charge_unconditional']:+.4f}, "
          f"partial={r['rho_charge_partial']:+.4f}, shift={r['shift_partial_minus_unconditional']:+.4f} "
          f"-> {label} ({note})")

partial_verdict_path = OUT / "partial_correlation_verdict.json"
with open(partial_verdict_path, "w") as f:
    json.dump(
        {
            "threshold_reused_from_part3": spearman_thresh,
            "per_isoform": {
                k: {
                    "part3_pass": v["part3_pass"],
                    "label": v["label"],
                    "note": v["note"],
                    "rho_charge_unconditional": v["rho_charge_unconditional"],
                    "rho_charge_partial": v["rho_charge_partial"],
                }
                for k, v in partial_verdicts.items()
            },
        },
        f, indent=2,
    )
print(f"\nsaved {partial_verdict_path}")
print("(outputs/17_protonation_charge_check/verdict_summary.json left untouched)")

CYP1A2: unconditional=-0.2047, partial=-0.1931, shift=+0.0116 -> SURVIVES (partial |rho| stays >= threshold, same sign -- independent of logP)
CYP2C9: unconditional=-0.1535, partial=-0.1204, shift=+0.0331 -> SURVIVES (partial |rho| stays >= threshold, same sign -- independent of logP)
CYP2D6: unconditional=+0.1691, partial=+0.1772, shift=+0.0081 -> SURVIVES (partial |rho| stays >= threshold, same sign -- independent of logP)
CYP3A4: unconditional=-0.0712, partial=-0.0029, shift=+0.0683 -> not_applicable (already FAILED Part 3's unconditional bar -- reported for completeness only)

saved /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/17_protonation_charge_check/partial_correlation_verdict.json
(outputs/17_protonation_charge_check/verdict_summary.json left untouched)


### Result, stated plainly — not fitted to the a-priori expectation after the fact

- **CYP1A2: SURVIVES** (unconditional -0.2047 → partial -0.1931, shift only +0.0116 — the
  charge/logP confound explains barely 6% of the magnitude of CYP1A2's raw correlation). This
  isoform's own charge-vs-logP correlation on its subset is weak (-0.075), so there was never much
  shared variance for logP to absorb. **This is a real, somewhat surprising result relative to
  Part 3's own qualitative hedge** — the hedge treated CYP1A2's signal as "not distinguishable
  from" the confound; quantified, it mostly is distinguishable. This was one of the two
  genuinely-uncertain isoforms the pre-registration above allowed to go either way, and it landed
  on SURVIVES.
- **CYP2C9: SURVIVES** (unconditional -0.1535 → partial -0.1204, shift +0.0331 — a real dent,
  about 22% of the raw magnitude, but not enough to cross the 0.10 floor). Also one of the
  genuinely-uncertain isoforms; also lands on SURVIVES, though with a visibly larger confound
  contribution than CYP1A2's.
- **CYP2D6: SURVIVES**, exactly as pre-registered (unconditional +0.1691 → partial +0.1772 — the
  partial correlation is if anything *slightly larger* than the raw one, since logP carries
  essentially no relationship to CYP2D6 pIC50 for the confound to act through). This remains the
  cleanest single strand of evidence in this project's own data for the electrostatic mechanism
  this notebook set out to test.
- **CYP3A4: not_applicable, reported for completeness only** — it already failed Part 3's
  unconditional bar (-0.0712) and is not re-adjudicated with either label. For context, not as a
  verdict: its partial correlation collapses to essentially zero (-0.0029, p=0.89, a ~96% drop in
  magnitude) — consistent with, though not proof of, its already-weak raw signal being almost
  entirely a lipophilicity artifact rather than a near-miss on an independent effect.

**All three isoforms that passed Part 3's frozen criteria retain a real, independent-of-logP
charge signal.** The confound Part 3 raised as a caveat on CYP1A2/CYP2C9 turns out, quantified, to
account for only a small fraction of either isoform's raw correlation — real, present, and worth
having checked, but not the dominant explanation either qualitative read might have suggested.

### What this changes, and what it does not

**Part 3's frozen PASS verdicts for CYP1A2, CYP2C9, and CYP2D6 are not retracted.** They were
correct against the criteria frozen at the time, which tested only the unconditional Spearman
correlation and did not include a confound control — nothing about that evaluation was wrong.
**What changes is the interpretation carried forward into any later modelling decision**: this
section shows that interpretation should now lean toward all three PASS isoforms' charge signals
being largely real and not merely a lipophilicity artifact, where Part 3's own prose had left
CYP1A2 and CYP2C9's status more open. CYP3A4's FAIL verdict is unaffected and not revisited.

**Effect-size caution, so this is not over-read in either direction**: CYP2D6's partial rho
(~0.177) corresponds to roughly the same modest magnitude as Part 3's own +1-vs-0 median pIC50
difference for that isoform (4.8296 vs. 4.6631, a difference of ~0.17 log units) — a real,
logP-independent signal, but a small one, against CYP2D6's actual blind-submission ST-RAE of
0.8299 (notebook 12, per `docs/leaderboard_submissions.md`). The signal being genuine does not
make it large, and nothing here predicts how much (if at all) it would move a full multitask
Chemprop model's OOF ST-RAE if added as a descriptor.

**No adoption or modelling recommendation is made here.** Whether to build a charge-augmented
Chemprop run remains a separate task, left to the user. This section is read-only analysis: no
protonation, charge, or logP value was recomputed, no model was trained, and nothing was
submitted. `data/folds/cv_folds.csv`, `outputs/05_cv_comparison/`, every existing submission
artifact, Parts 1-5 above, Part 3's frozen `decision_criteria.json` block, and
`verdict_summary.json` are all untouched.